In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports ===

%matplotlib inline

import os
import sys
import json
import math
import random
import platform
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import yaml

import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import ultralytics
from ultralytics import YOLO

import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10


In [ ]:
# === Cell 3: Config ===

from pprint import pprint

# --- Project paths ---
EXPORT_ROOT    = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export")
TRAIN_ROOT     = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260529_Yolo_Final_Training_yolov8n")
BEST_HPO       = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260528_Yolo_Hyperparameter_Tuning_yolov8n/results/20260528_Yolo_Hyperparameter_Tuning_yolov8n_best_trial_hyperparameters.csv")
HPO_ROOT       = BEST_HPO.parents[1]
SCREENING_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260527_Yolo_Model_Screening")

IMAGES_DIR = EXPORT_ROOT / "images" / "all"
LABELS_DIR = EXPORT_ROOT / "labels" / "all"

OBJECT_SPLIT_CSV = EXPORT_ROOT / "object_split.csv"
METADATA_AGG_CSV = EXPORT_ROOT / "metadata_aggregated.csv"
BASE_DATA_YAML   = EXPORT_ROOT / "data.yaml"

# --- Output folders ---
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS_DIR    = TRAIN_ROOT / "splits"
RUNS_DIR      = TRAIN_ROOT / "runs"
RESULTS_DIR   = TRAIN_ROOT / "results"
INFERENCE_DIR = TRAIN_ROOT / "inference"

for p in [SPLITS_DIR, RUNS_DIR, RESULTS_DIR, INFERENCE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME    = TRAIN_ROOT.name
EXPERIMENT_NAME = "20260529_Yolo_Final_Training_yolov8n"
RUN_NAME        = f"{PROJECT_NAME}_{EXPERIMENT_NAME}"

# --- Dataset semantics ---
CLASS_MAP = {
    "body":  0,
    "cap":   1,
    "label": 2,
}
MATERIAL_CLASSES = ["HDPE", "PET", "PP", "PS"]

# --- Reproducibility and execution ---
RANDOM_SEED     = 42
DEVICE          = 0
WORKERS         = 2
ALLOW_OVERWRITE = True

# --- Final model selection ---
SELECTED_MODEL = "yolov8n-seg.pt"
TASK           = "segment"

# --- Final training setup ---
TRAIN_EPOCHS             = 300
TRAIN_PATIENCE           = 50
BATCH_SIZE               = 16
IMAGE_SIZE               = 640
IMAGE_RECT               = True
SAVE_PLOTS               = True
PRETRAINED               = True
VERBOSE                  = True
VALIDATE_DURING_TRAINING = True

NUM_FOLDS            = 4
FINAL_VAL_FOLD_INDEX = 0
FOLDS_TO_RUN         = [FINAL_VAL_FOLD_INDEX]

# --- Inference and evaluation setup ---
CONF_THRESHOLD = 0.50
IOU_THRESHOLD  = 0.50
MAX_DET        = 300

BEST_EPOCH_METRIC           = "metrics/mAP50-95(M)"
BEST_MODEL_SELECTION_METRIC = "metrics/mAP50-95(M)"
F1_METRIC_NAME              = "metrics/F1(M)"

# --- HPO parsing ---
BEST_HPO_DF = pd.read_csv(BEST_HPO)
BEST_HPO_DF.columns = [str(c).strip() for c in BEST_HPO_DF.columns]

required_cols = {"trial_id", "hyperparameter", "value"}
missing_cols  = required_cols - set(BEST_HPO_DF.columns)
if missing_cols:
    raise ValueError(f"BEST_HPO missing columns: {missing_cols}")

BEST_TRIAL_ID = BEST_HPO_DF["trial_id"].iloc[0]

# Maps HPO CSV names (with or without underscores) to Ultralytics train() keys.
HPO_TO_YOLO_KEY_MAP = {
    "lr0":             "lr0",
    "lrf":             "lrf",
    "momentum":        "momentum",
    "weight_decay":    "weight_decay",
    "weightdecay":     "weight_decay",
    "warmup_epochs":   "warmup_epochs",
    "warmupepochs":    "warmup_epochs",
    "warmup_momentum": "warmup_momentum",
    "warmupmomentum":  "warmup_momentum",
    "box":             "box",
    "cls":             "cls",
    "dfl":             "dfl",
    "hsv_h":           "hsv_h",
    "hsvh":            "hsv_h",
    "hsv_s":           "hsv_s",
    "hsvs":            "hsv_s",
    "hsv_v":           "hsv_v",
    "hsvv":            "hsv_v",
    "degrees":         "degrees",
    "translate":       "translate",
    "scale":           "scale",
    "shear":           "shear",
    "perspective":     "perspective",
    "flipud":          "flipud",
    "fliplr":          "fliplr",
    "bgr":             "bgr",
    "mosaic":          "mosaic",
    "mixup":           "mixup",
    "copy_paste":      "copy_paste",
    "copypaste":       "copy_paste",
    "cutmix":          "cutmix",
}

VALID_YOLO_HPARAMS = {
    "lr0", "lrf", "momentum", "weight_decay", "warmup_epochs", "warmup_momentum",
    "box", "cls", "dfl", "hsv_h", "hsv_s", "hsv_v", "degrees", "translate",
    "scale", "shear", "perspective", "flipud", "fliplr", "bgr", "mosaic",
    "mixup", "copy_paste", "cutmix",
}

# Converts CSV strings to int, float or bool where possible.
def normalize_scalar(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, str):
        value = value.strip()
        try:
            f = float(value)
            return int(f) if f.is_integer() else f
        except Exception:
            if value.lower() in {"true", "false"}:
                return value.lower() == "true"
            return value
    return value

def build_best_hpo_dict(hpo_df):
    best_dict = {}
    ignored   = []
    for _, row in hpo_df.iterrows():
        raw_key    = str(row["hyperparameter"]).strip()
        raw_value  = normalize_scalar(row["value"])
        mapped_key = HPO_TO_YOLO_KEY_MAP.get(raw_key)
        if mapped_key is None or mapped_key not in VALID_YOLO_HPARAMS:
            ignored.append(raw_key)
            continue
        best_dict[mapped_key] = raw_value
    return best_dict, sorted(set(ignored))

BEST_HPO_HPARAMS, IGNORED_HPO_KEYS = build_best_hpo_dict(BEST_HPO_DF)

# Optional overrides applied on top of the best HPO trial.
MANUAL_HPARAM_OVERRIDES = {
    # "lr0": 0.001,
    # "mosaic": 0.5,
}

FINAL_TRAIN_HPARAMS = dict(BEST_HPO_HPARAMS)
FINAL_TRAIN_HPARAMS.update(MANUAL_HPARAM_OVERRIDES)

# --- Export paths ---
FINAL_HPARAM_CSV        = RESULTS_DIR / f"{PROJECT_NAME}_final_hyperparameters.csv"
FINAL_TRAIN_KWARGS_JSON = RESULTS_DIR / f"{PROJECT_NAME}_final_train_kwargs.json"
FINAL_RUN_SUMMARY_JSON  = RESULTS_DIR / f"{PROJECT_NAME}_final_run_summary.json"
FINAL_EVAL_CSV          = RESULTS_DIR / f"{PROJECT_NAME}_final_test_metrics.csv"
FINAL_EVAL_JSON         = RESULTS_DIR / f"{PROJECT_NAME}_final_test_metrics.json"
PER_IMAGE_EVAL_CSV      = RESULTS_DIR / f"{PROJECT_NAME}_per_image_evaluation.csv"
VISUALIZATION_PNG       = RESULTS_DIR / f"{PROJECT_NAME}_final_visualization_examples.png"
FINAL_INFERENCE_JSON    = INFERENCE_DIR / f"{PROJECT_NAME}_inference_predictions.json"
FINAL_INFERENCE_CSV     = INFERENCE_DIR / f"{PROJECT_NAME}_inference_index.csv"

# --- Deterministic seeds ---
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Configuration")
print("-" * 80)
print("EXPORT_ROOT:                ", EXPORT_ROOT)
print("TRAIN_ROOT:                 ", TRAIN_ROOT)
print("HPO_ROOT:                   ", HPO_ROOT)
print("SCREENING_ROOT:             ", SCREENING_ROOT)
print("BEST_HPO:                   ", BEST_HPO)
print("SELECTED_MODEL:             ", SELECTED_MODEL)
print("RUN_NAME:                   ", RUN_NAME)
print("DEVICE:                     ", DEVICE)
print("NUM_FOLDS:                  ", NUM_FOLDS)
print("FINAL_VAL_FOLD_INDEX:       ", FINAL_VAL_FOLD_INDEX)
print("BEST_TRIAL_ID:              ", BEST_TRIAL_ID)
print("IGNORED_HPO_KEYS:           ", IGNORED_HPO_KEYS)
print("BEST_EPOCH_METRIC:          ", BEST_EPOCH_METRIC)
print("BEST_MODEL_SELECTION_METRIC:", BEST_MODEL_SELECTION_METRIC)
print("FINAL_TRAIN_HPARAMS:")
pprint(FINAL_TRAIN_HPARAMS)

pd.DataFrame([
    {"hyperparameter": k, "value": v} for k, v in FINAL_TRAIN_HPARAMS.items()
]).sort_values("hyperparameter").to_csv(FINAL_HPARAM_CSV, index=False)

print("\nSaved final hyperparameters to:", FINAL_HPARAM_CSV)

In [ ]:
# === Cell 4: Sanity Check ===

print("Python:            ", sys.version)
print("Platform:          ", platform.platform())
print("torch version:     ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count: ", torch.cuda.device_count())
    print("CUDA device 0:     ", torch.cuda.get_device_name(0))
print("ultralytics version:", ultralytics.__version__)

PATHS_TO_CHECK = [
    ("EXPORT_ROOT",      EXPORT_ROOT),
    ("TRAIN_ROOT",       TRAIN_ROOT),
    ("HPO_ROOT",         HPO_ROOT),
    ("SCREENING_ROOT",   SCREENING_ROOT),
    ("IMAGES_DIR",       IMAGES_DIR),
    ("LABELS_DIR",       LABELS_DIR),
    ("OBJECT_SPLIT_CSV", OBJECT_SPLIT_CSV),
    ("METADATA_AGG_CSV", METADATA_AGG_CSV),
    ("BASE_DATA_YAML",   BASE_DATA_YAML),
    ("BEST_HPO",         BEST_HPO),
    ("SPLITS_DIR",       SPLITS_DIR),
    ("RUNS_DIR",         RUNS_DIR),
    ("RESULTS_DIR",      RESULTS_DIR),
    ("INFERENCE_DIR",    INFERENCE_DIR),
]

all_paths_ok = True
print("\nPath checks")
for path_name, path_value in PATHS_TO_CHECK:
    exists = path_value.exists()
    kind   = "DIR " if path_value.is_dir() else "FILE"
    status = "OK     " if exists else "MISSING"
    print(f"  [{kind}] {status}  {path_name}: {path_value}")
    if path_name in {"EXPORT_ROOT", "IMAGES_DIR", "LABELS_DIR",
                     "OBJECT_SPLIT_CSV", "BASE_DATA_YAML", "BEST_HPO"} and not exists:
        all_paths_ok = False

if FINAL_TRAIN_KWARGS_JSON.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError(
        f"Refusing to overwrite existing config: {FINAL_TRAIN_KWARGS_JSON}. "
        "Set ALLOW_OVERWRITE=True to continue."
    )

objectsplitdf = None
imagepaths    = list(IMAGES_DIR.glob("*")) if IMAGES_DIR.exists() else []

if OBJECT_SPLIT_CSV.exists():
    objectsplitdf = pd.read_csv(OBJECT_SPLIT_CSV, dtype={"object_id": str})
    
    # Strip whitespace from column names.
    objectsplitdf.columns = [c.strip() for c in objectsplitdf.columns]
    
    # Show the CSV columns as a quick schema check.
    print("\nobject_split.csv columns:", list(objectsplitdf.columns))
    
    uniqueobjects = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    print("\nDataset stats")
    print("  Distinct objects:  ", len(uniqueobjects))
    print("  Distinct materials:", uniqueobjects["material"].nunique())
    print("  Total images:      ", len(imagepaths))
    display(uniqueobjects.groupby("material")["object_id"].nunique().rename("n_objects"))
else:
    raise FileNotFoundError(f"object_split.csv not found: {OBJECT_SPLIT_CSV}")

try:
    YOLO(SELECTED_MODEL)
    print(f"\nModel instantiation OK: {SELECTED_MODEL}")
except Exception as exc:
    raise RuntimeError(f"Model instantiation failed for {SELECTED_MODEL}") from exc

if not all_paths_ok:
    raise RuntimeError("One or more required paths are missing. Fix config before continuing.")

print("\nSanity check PASS")

In [ ]:
# === Cell 5: Filename Parsing ===

# Expected export filename format: HDPE_obj_01_per_0_2026-04-21.png
# Images are grouped by (material, object_id) for the object-level split.

FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)"
    r"_obj_(?P<objectid>\d+)"
    r"_per_(?P<perspectiveid>\d+)"
    r"_(?P<date>\d{4}-\d{2}-\d{2})"
    r"\.(?P<extension>png|jpg|jpeg)$",
    re.IGNORECASE,
)

def parse_material_object_perspective(filename: str):
    match = FILENAME_REGEX.match(filename)
    if not match:
        raise ValueError(f"Filename does not match expected pattern: {filename}")
    material      = match.group("material")
    objectid      = match.group("objectid")
    perspectiveid = match.group("perspectiveid")
    datestr       = match.group("date")
    return material, objectid, perspectiveid, datestr

object_to_images = defaultdict(list)
bad_filenames    = []

for image_path in imagepaths:
    filename = image_path.name
    try:
        material, objectid, perspectiveid, datestr = parse_material_object_perspective(filename)
        object_to_images[(material, objectid)].append(image_path)
    except ValueError:
        bad_filenames.append(filename)

print("Mapped objects from filenames:", len(object_to_images))

if bad_filenames:
    print(f"\nWARNING: {len(bad_filenames)} filenames did not match expected pattern:")
    for bad_name in bad_filenames[:10]:
        print("  -", bad_name)
    if len(bad_filenames) > 10:
        print(f"  ... and {len(bad_filenames) - 10} more")

if objectsplitdf is not None:
    uniqueobjects   = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    split_objects   = set(zip(uniqueobjects["material"], uniqueobjects["object_id"]))
    mapped_objects  = set(object_to_images.keys())

    missing_in_images = split_objects - mapped_objects
    extra_in_images   = mapped_objects - split_objects

    print("\nConsistency checks")
    print("  Objects in object_split.csv but missing in images:", len(missing_in_images))
    print("  Objects in images but missing in object_split.csv:", len(extra_in_images))

    if missing_in_images:
        print("  Example missing:", list(missing_in_images)[:10])
    if extra_in_images:
        print("  Example extra:  ", list(extra_in_images)[:10])

    if len(missing_in_images) > 0:
        print("\nWARNING: some objects in the split CSV have no images — check IMAGES_DIR path.")
    else:
        print("\nFilename mapping OK — all split objects have images.")

In [ ]:
# === Cell 6: CV Fold Builder ===

# Builds object-level CV folds per material (train/val lists and data YAMLs), the
# test list, and the final split: train/val of the reference fold plus the test set.

random.seed(RANDOM_SEED)

cvdf      = objectsplitdf[objectsplitdf["split"] == "cv"].copy()
testdf    = objectsplitdf[objectsplitdf["split"] == "test"].copy()
skippeddf = objectsplitdf[objectsplitdf["split"] == "skipped"].copy()

def objects_to_image_paths(object_keys):
    paths = []
    for material, object_id in object_keys:
        images = object_to_images.get((material, object_id), [])
        if not images:
            print(f"WARNING: no images found for object {(material, object_id)}")
        paths.extend(images)
    return sorted(paths)

def build_split_summary_df(df, split_name):
    rows = []
    if df.empty:
        return pd.DataFrame(columns=["split", "material", "objects", "images"])
    for material in sorted(df["material"].unique()):
        material_df   = df[df["material"] == material]
        material_keys = list(zip(material_df["material"], material_df["object_id"]))
        rows.append({
            "split":    split_name,
            "material": material,
            "objects":  material_df["object_id"].nunique(),
            "images":   len(objects_to_image_paths(material_keys)),
        })
    summary_df = pd.DataFrame(rows)
    total_row  = pd.DataFrame([{
        "split":    split_name,
        "material": "TOTAL",
        "objects":  summary_df["objects"].sum(),
        "images":   summary_df["images"].sum(),
    }])
    return pd.concat([summary_df, total_row], ignore_index=True)

print("Split summary by material")
split_summary_df = pd.concat([
    build_split_summary_df(cvdf,      "CV"),
    build_split_summary_df(testdf,    "Test"),
    build_split_summary_df(skippeddf, "Skipped"),
], ignore_index=True)
display(split_summary_df)

test_object_keys = list(zip(testdf["material"], testdf["object_id"]))
test_images      = objects_to_image_paths(test_object_keys)

TESTTXT = SPLITS_DIR / "test.txt"
with TESTTXT.open("w", encoding="utf-8") as f:
    for image_path in test_images:
        f.write(str(image_path.resolve()) + "\n")

# Shuffles object IDs and partitions them into num_folds near-equal groups.
def make_folds_for_material(objectids, num_folds=NUM_FOLDS):
    objectids = list(objectids)
    random.shuffle(objectids)

    num_total  = len(objectids)
    if num_total == 0:
        return [{"train": [], "val": []} for _ in range(num_folds)]

    fold_sizes = [num_total // num_folds] * num_folds
    for i in range(num_total % num_folds):
        fold_sizes[i] += 1

    partitions, start = [], 0
    for fold_size in fold_sizes:
        end = start + fold_size
        partitions.append(objectids[start:end])
        start = end

    fold_defs = []
    for fold_index in range(num_folds):
        val_ids   = partitions[fold_index]
        train_ids = [obj for i, part in enumerate(partitions) if i != fold_index for obj in part]
        fold_defs.append({"train": train_ids, "val": val_ids})
    return fold_defs

materials         = sorted(cvdf["material"].unique())
folds             = defaultdict(dict)
material_fold_rows = []

for material in materials:
    material_cv_df    = cvdf[cvdf["material"] == material]
    material_objectids = sorted(material_cv_df["object_id"].unique())
    material_folds    = make_folds_for_material(material_objectids, num_folds=NUM_FOLDS)

    for fold_index, fold_def in enumerate(material_folds):
        train_keys_local = [(material, oid) for oid in fold_def["train"]]
        val_keys_local   = [(material, oid) for oid in fold_def["val"]]

        material_fold_rows.append({
            "fold":         f"fold{fold_index + 1}",
            "material":     material,
            "trainobjects": len(fold_def["train"]),
            "trainimages":  len(objects_to_image_paths(train_keys_local)),
            "valobjects":   len(fold_def["val"]),
            "valimages":    len(objects_to_image_paths(val_keys_local)),
        })

        folds[fold_index][material] = fold_def

print("Per-material fold summary")
display(pd.DataFrame(material_fold_rows))

with open(BASE_DATA_YAML, "r", encoding="utf-8") as f:
    base_data = yaml.safe_load(f)

FOLDCONFIGS      = {}
fold_summary_rows = []

for fold_index in range(NUM_FOLDS):
    fold_name = f"fold{fold_index + 1}"
    fold_dir  = SPLITS_DIR / fold_name
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_keys, val_keys = [], []
    for material in materials:
        train_keys.extend((material, oid) for oid in folds[fold_index][material]["train"])
        val_keys.extend((material, oid)   for oid in folds[fold_index][material]["val"])

    train_images = objects_to_image_paths(train_keys)
    val_images   = objects_to_image_paths(val_keys)

    train_txt = fold_dir / "train.txt"
    val_txt   = fold_dir / "val.txt"

    with train_txt.open("w", encoding="utf-8") as f:
        for image_path in train_images:
            f.write(str(image_path.resolve()) + "\n")

    with val_txt.open("w", encoding="utf-8") as f:
        for image_path in val_images:
            f.write(str(image_path.resolve()) + "\n")

    fold_data          = dict(base_data)
    fold_data["path"]  = str(EXPORT_ROOT.resolve())
    fold_data["train"] = str(train_txt.resolve())
    fold_data["val"]   = str(val_txt.resolve())
    fold_data["test"]  = str(TESTTXT.resolve())

    fold_yaml = SPLITS_DIR / f"data_{fold_name}.yaml"
    with fold_yaml.open("w", encoding="utf-8") as f:
        yaml.safe_dump(fold_data, f, sort_keys=False)

    FOLDCONFIGS[fold_index] = {
        "name":            fold_name,
        "traintxt":        train_txt,
        "valtxt":          val_txt,
        "testtxt":         TESTTXT,
        "datayaml":        fold_yaml,
        "numtrainimages":  len(train_images),
        "numvalimages":    len(val_images),
        "numtestimages":   len(test_images),
        "numtrainobjects": len(train_keys),
        "numvalobjects":   len(val_keys),
    }

    fold_summary_rows.append({
        "fold":         fold_name,
        "trainobjects": len(train_keys),
        "trainimages":  len(train_images),
        "valobjects":   len(val_keys),
        "valimages":    len(val_images),
        "testobjects":  len(test_object_keys),
        "testimages":   len(test_images),
        "datayaml":     str(fold_yaml),
    })

print("Fold summary")
display(pd.DataFrame(fold_summary_rows))

if FINAL_VAL_FOLD_INDEX not in FOLDCONFIGS:
    raise KeyError(f"FINAL_VAL_FOLD_INDEX={FINAL_VAL_FOLD_INDEX} not found in FOLDCONFIGS")

# --- Final split ---
FINAL_FOLD_CONFIG = FOLDCONFIGS[FINAL_VAL_FOLD_INDEX]
FINAL_TRAIN_TXT   = SPLITS_DIR / "final_train.txt"
FINAL_VAL_TXT     = SPLITS_DIR / "final_val.txt"
FINAL_TEST_TXT    = TESTTXT
FINAL_DATA_YAML   = SPLITS_DIR / "data_final.yaml"
TEST_DATA_YAML    = INFERENCE_DIR / "data_test.yaml"

final_train_images = [Path(x.strip()) for x in FINAL_FOLD_CONFIG["traintxt"].read_text(encoding="utf-8").splitlines() if x.strip()]
final_val_images   = [Path(x.strip()) for x in FINAL_FOLD_CONFIG["valtxt"].read_text(encoding="utf-8").splitlines() if x.strip()]
final_test_images  = [Path(x.strip()) for x in FINAL_TEST_TXT.read_text(encoding="utf-8").splitlines() if x.strip()]

with FINAL_TRAIN_TXT.open("w", encoding="utf-8") as f:
    for p in final_train_images:
        f.write(str(p.resolve()) + "\n")

with FINAL_VAL_TXT.open("w", encoding="utf-8") as f:
    for p in final_val_images:
        f.write(str(p.resolve()) + "\n")

final_data          = dict(base_data)
final_data["path"]  = str(EXPORT_ROOT.resolve())
final_data["train"] = str(FINAL_TRAIN_TXT.resolve())
final_data["val"]   = str(FINAL_VAL_TXT.resolve())
final_data["test"]  = str(FINAL_TEST_TXT.resolve())
with FINAL_DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(final_data, f, sort_keys=False)

# Test YAML uses the test list for all splits (for model.val on the test set).
test_data          = dict(base_data)
test_data["path"]  = str(EXPORT_ROOT.resolve())
test_data["train"] = str(FINAL_TEST_TXT.resolve())
test_data["val"]   = str(FINAL_TEST_TXT.resolve())
test_data["test"]  = str(FINAL_TEST_TXT.resolve())
with TEST_DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(test_data, f, sort_keys=False)

print("\nFinal training split summary")
print("Reference fold:   ", FINAL_FOLD_CONFIG["name"])
print("FINAL_TRAIN_TXT:  ", FINAL_TRAIN_TXT)
print("FINAL_VAL_TXT:    ", FINAL_VAL_TXT)
print("FINAL_TEST_TXT:   ", FINAL_TEST_TXT)
print("FINAL_DATA_YAML:  ", FINAL_DATA_YAML)
print("TEST_DATA_YAML:   ", TEST_DATA_YAML)
print("#train images:    ", len(final_train_images))
print("#val images:      ", len(final_val_images))
print("#test images:     ", len(final_test_images))

In [ ]:
# === Cell 7: Final Training ===

# Splits e.g. "yolov8n-seg.pt" into ("yolov8", "n").
def parse_model_version_size(model_name: str):
    stem = Path(model_name).stem
    base = stem.split("-")[0]
    if not base.startswith("yolo") or len(base) < 6:
        return base, None
    return base[:-1], base[-1]

def serialize_for_json(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, dict):
        return {k: serialize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [serialize_for_json(v) for v in obj]
    return obj

MODEL_VERSION, MODEL_SIZE = parse_model_version_size(SELECTED_MODEL)

train_kwargs = {
    "data": str(FINAL_DATA_YAML),
    "epochs": TRAIN_EPOCHS,
    "patience": TRAIN_PATIENCE,
    "batch": BATCH_SIZE,
    "imgsz": IMAGE_SIZE,
    "device": DEVICE,
    "workers": WORKERS,
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": ALLOW_OVERWRITE,
    "task": TASK,
    "rect": IMAGE_RECT,
    "seed": RANDOM_SEED,
    "deterministic": True,
    "val": VALIDATE_DURING_TRAINING,
    "plots": SAVE_PLOTS,
    "pretrained": PRETRAINED,
    "verbose": VERBOSE,
    "optimizer": "AdamW",
}
# Best HPO hyperparameters are added on top of the base settings.
train_kwargs.update(FINAL_TRAIN_HPARAMS)

with FINAL_TRAIN_KWARGS_JSON.open("w", encoding="utf-8") as f:
    json.dump(serialize_for_json(train_kwargs), f, indent=2, ensure_ascii=False)

print("Training kwargs")
pprint(train_kwargs)

final_model = YOLO(SELECTED_MODEL)
train_results = final_model.train(**train_kwargs)

# Run directory from the train results; falls back to RUNS_DIR / RUN_NAME.
if train_results is not None and hasattr(train_results, "save_dir"):
    FINAL_RUN_DIR = Path(train_results.save_dir)
elif train_results is not None and hasattr(train_results, "savedir"):
    FINAL_RUN_DIR = Path(train_results.savedir)
else:
    FINAL_RUN_DIR = RUNS_DIR / RUN_NAME

FINAL_WEIGHTS_BEST = FINAL_RUN_DIR / "weights" / "best.pt"
FINAL_WEIGHTS_LAST = FINAL_RUN_DIR / "weights" / "last.pt"
FINAL_RESULTS_CSV = FINAL_RUN_DIR / "results.csv"

run_summary = {
    "timestamp": datetime.now().isoformat(),
    "project_name": PROJECT_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "run_name": RUN_NAME,
    "selected_model": SELECTED_MODEL,
    "model_version": MODEL_VERSION,
    "model_size": MODEL_SIZE,
    "best_trial_id": int(BEST_TRIAL_ID) if pd.notna(BEST_TRIAL_ID) else None,
    "final_val_fold_index": FINAL_VAL_FOLD_INDEX,
    "final_val_fold_name": FINAL_FOLD_CONFIG["name"],
    "final_data_yaml": FINAL_DATA_YAML,
    "final_run_dir": FINAL_RUN_DIR,
    "weights_best": FINAL_WEIGHTS_BEST,
    "weights_last": FINAL_WEIGHTS_LAST,
    "results_csv": FINAL_RESULTS_CSV,
    "train_kwargs": train_kwargs,
}

with FINAL_RUN_SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(serialize_for_json(run_summary), f, indent=2, ensure_ascii=False)

print("Training finished")
print("FINAL_RUN_DIR:     ", FINAL_RUN_DIR)
print("FINAL_WEIGHTS_BEST:", FINAL_WEIGHTS_BEST, FINAL_WEIGHTS_BEST.exists())
print("FINAL_RESULTS_CSV: ", FINAL_RESULTS_CSV, FINAL_RESULTS_CSV.exists())
print("Run summary saved: ", FINAL_RUN_SUMMARY_JSON)


In [ ]:
# === Cell 8: Test-Set Inference ===

# Predicts on every test image and saves a per-image CSV index (sorted by count
# error) and a JSON file with classes, confidences, boxes and mask polygons.

if "FINAL_WEIGHTS_BEST" not in globals() or not FINAL_WEIGHTS_BEST.exists():
    raise FileNotFoundError("Best weights not found. Run Cell 7 first.")

def read_non_empty_lines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

# Parses a YOLO segmentation label line: class_id followed by normalised x y pairs.
def parse_yolo_label_line(line):
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    cls_id = int(float(parts[0]))
    coords = [float(v) for v in parts[1:]]
    polygon = list(zip(coords[0::2], coords[1::2]))
    return {"class_id": cls_id, "polygon_norm": polygon}

def count_gt_instances_for_image(image_path):
    label_path = LABELS_DIR / (Path(image_path).stem + ".txt")
    if not label_path.exists():
        return 0, label_path
    with label_path.open("r", encoding="utf-8") as f:
        gt_rows = [parse_yolo_label_line(line) for line in f if line.strip()]
    gt_rows = [row for row in gt_rows if row is not None]
    return len(gt_rows), label_path

TEST_IMAGE_PATHS = [Path(p) for p in read_non_empty_lines(FINAL_TEST_TXT)]
print(f"Running inference on {len(TEST_IMAGE_PATHS)} test images")

inference_model = YOLO(str(FINAL_WEIGHTS_BEST))
prediction_rows = []
json_payload = []

for image_path in TEST_IMAGE_PATHS:
    results = inference_model.predict(
        source=str(image_path),
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMAGE_SIZE,
        max_det=MAX_DET,
        retina_masks=True,
        verbose=False,
        save=False,
    )
    result = results[0]
    boxes = result.boxes
    masks = result.masks

    if boxes is None or len(boxes) == 0:
        cls_ids = []
        confs = []
        xyxy = []
    else:
        cls_ids = boxes.cls.cpu().numpy().astype(int).tolist()
        confs = boxes.conf.cpu().numpy().astype(float).tolist()
        xyxy = boxes.xyxy.cpu().numpy().tolist()

    polygons = masks.xy if masks is not None and masks.xy is not None else []
    polygons = [np.asarray(poly).tolist() for poly in polygons]

    gt_instances, label_path = count_gt_instances_for_image(image_path)

    prediction_rows.append({
        "image_path": str(image_path),
        "image_name": image_path.name,
        "label_path": str(label_path),
        "pred_instances": len(cls_ids),
        "gt_instances": gt_instances,
        "abs_count_error": abs(len(cls_ids) - gt_instances),
        "mean_conf": float(np.mean(confs)) if len(confs) else np.nan,
        "max_conf": float(np.max(confs)) if len(confs) else np.nan,
        "class_ids": json.dumps(cls_ids),
        "confidences": json.dumps(confs),
        "boxes_xyxy": json.dumps(xyxy),
    })

    json_payload.append({
        "image_path": str(image_path),
        "label_path": str(label_path),
        "class_ids": cls_ids,
        "confidences": confs,
        "boxes_xyxy": xyxy,
        "polygons_xy": polygons,
        "gt_instances": gt_instances,
    })

inference_index_df = pd.DataFrame(prediction_rows).sort_values(
    ["abs_count_error", "image_name"], ascending=[False, True]
).reset_index(drop=True)

inference_index_df.to_csv(FINAL_INFERENCE_CSV, index=False)
with FINAL_INFERENCE_JSON.open("w", encoding="utf-8") as f:
    json.dump(json_payload, f, indent=2, ensure_ascii=False)

print("Saved inference index CSV to:", FINAL_INFERENCE_CSV)
print("Saved inference JSON to:     ", FINAL_INFERENCE_JSON)
display(inference_index_df.head(10))


In [ ]:
# === Cell 9: Test Metrics and Plots ===

if "FINAL_WEIGHTS_BEST" not in globals() or not FINAL_WEIGHTS_BEST.exists():
    raise FileNotFoundError("Best weights not found. Run Cell 7 first.")

if "FINAL_RUN_DIR" not in globals():
    FINAL_RUN_DIR = RUNS_DIR / RUN_NAME

FINAL_RESULTS_CSV = FINAL_RUN_DIR / "results.csv"

# --- Helpers ---
# Metric attributes are methods or plain arrays depending on the Ultralytics version.
def _safe_get(obj, *attrs, default=np.nan):
    try:
        cur = obj
        for a in attrs:
            cur = getattr(cur, a)
            if callable(cur) and not isinstance(cur, np.ndarray):
                cur = cur()
        return cur
    except Exception:
        return default

def _safe_float(obj, *attrs, default=np.nan):
    val = _safe_get(obj, *attrs, default=default)
    try:    return float(val)
    except: return default

def _find_col(df, patterns):
    """First column whose name contains any pattern (case-insensitive)."""
    for p in patterns:
        for c in df.columns:
            if p.lower() in c.lower():
                return c
    return None

# --- Training curves from Ultralytics results.csv ---
if not FINAL_RESULTS_CSV.exists():
    print(f"WARNING: results.csv not found at {FINAL_RESULTS_CSV} — skipping curves.")
else:
    tdf = pd.read_csv(FINAL_RESULTS_CSV)
    tdf.columns = [c.strip() for c in tdf.columns]
    ep  = pd.to_numeric(tdf[_find_col(tdf, ["epoch"]) or tdf.columns[0]], errors="coerce")

    # --- Loss curves ---
    loss_specs = [
        ("Train box loss",  ["train/box", "box_loss"]),
        ("Train seg loss",  ["train/seg", "seg_loss"]),
        ("Train cls loss",  ["train/cls", "cls_loss"]),
        ("Train dfl loss",  ["train/dfl", "dfl_loss"]),
        ("Val box loss",    ["val/box"]),
        ("Val seg loss",    ["val/seg"]),
        ("Val cls loss",    ["val/cls"]),
        ("Val dfl loss",    ["val/dfl"]),
    ]
    fig_l, axes_l = plt.subplots(2, 4, figsize=(18, 7))
    fig_l.suptitle(f"Loss curves — {RUN_NAME}", fontsize=12)
    for ax, (lbl, pats) in zip(axes_l.reshape(-1), loss_specs):
        col = _find_col(tdf, pats)
        clr = "tab:blue" if "Train" in lbl else "tab:orange"
        if col:
            ax.plot(ep, pd.to_numeric(tdf[col], errors="coerce"), color=clr)
            ax.set_title(lbl, fontsize=9)
            ax.set_xlabel("Epoch", fontsize=8)
            ax.grid(alpha=0.3)
        else:
            ax.set_title(f"{lbl}\n(not found)", fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    loss_png = RESULTS_DIR / f"{PROJECT_NAME}_loss_curves.png"
    fig_l.savefig(loss_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", loss_png)

    # --- Validation metric curves ---
    metric_specs = [
        ("Precision",  ["precision(M)", "precision"]),
        ("Recall",     ["recall(M)",    "recall"]),
        ("mAP@50",     ["mAP50(M)",     "mAP50"]),
        ("mAP@50-95",  ["mAP50-95(M)", "mAP50-95"]),
    ]
    fig_m, axes_m = plt.subplots(1, 4, figsize=(18, 4))
    fig_m.suptitle(f"Validation metrics per epoch — {RUN_NAME}", fontsize=12)
    for ax, (lbl, pats) in zip(axes_m, metric_specs):
        col = _find_col(tdf, pats)
        if col:
            vals = pd.to_numeric(tdf[col], errors="coerce")
            ax.plot(ep, vals, color="tab:green")
            best_i = vals.idxmax()
            bv, be = vals[best_i], int(ep[best_i])
            ax.axvline(be, color="red", linestyle="--", alpha=0.6,
                       label=f"best={bv:.3f} @ep{be}")
            ax.set_title(lbl); ax.set_xlabel("Epoch")
            ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        else:
            ax.set_title(f"{lbl}\n(not found)"); ax.axis("off")
    plt.tight_layout()
    metric_png = RESULTS_DIR / f"{PROJECT_NAME}_metric_curves.png"
    fig_m.savefig(metric_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", metric_png)

# --- model.val() on the test set ---
print("\nRunning model.val() on test set …")
eval_model   = YOLO(str(FINAL_WEIGHTS_BEST))
eval_results = eval_model.val(
    data=str(TEST_DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    device=DEVICE,
    workers=WORKERS,
    rect=IMAGE_RECT,
    plots=True,
    verbose=True,
    project=str(RUNS_DIR),   # outputs in RUNS_DIR / "val_test"
    name="val_test",
    exist_ok=ALLOW_OVERWRITE,
)

# --- Aggregate mask metrics ---
seg = _safe_get(eval_results, "seg", default=None)

prec  = _safe_float(seg, "mp")
rec   = _safe_float(seg, "mr")
map50 = _safe_float(seg, "map50")
map_  = _safe_float(seg, "map")
f1    = (2 * prec * rec / (prec + rec)) if (not np.isnan(prec + rec) and (prec + rec) > 0) else np.nan

agg_metrics = {
    "precision(M)": prec,
    "recall(M)":    rec,
    "mAP50(M)":     map50,
    "mAP50-95(M)":  map_,
    "F1(M)":        f1,
}
print("\nTest-set aggregate metrics (mask):")
display(pd.DataFrame([agg_metrics]).round(4))

# --- Per-class mask metrics (from all_ap) ---
per_class_rows = []
try:
    names         = eval_results.names                           # {0: 'body', …}
    ap_class_idx  = np.asarray(_safe_get(seg, "ap_class_index")) # [n_classes]
    all_ap        = np.asarray(_safe_get(seg, "all_ap"))         # [n_classes, n_iou]

    if all_ap.ndim == 2 and len(ap_class_idx) > 0:
        ap50_pc   = all_ap[:, 0]                # AP @ IoU=0.50
        ap5095_pc = all_ap.mean(axis=1)         # mean AP @ IoU=0.50:0.95

        # Per-class precision and recall
        p_pc = np.asarray(_safe_get(seg, "p", default=np.full(len(ap_class_idx), np.nan)))
        r_pc = np.asarray(_safe_get(seg, "r", default=np.full(len(ap_class_idx), np.nan)))

        for i, cls_idx in enumerate(ap_class_idx):
            cls_name = names.get(int(cls_idx), str(cls_idx))
            p_i = float(p_pc[i]) if i < len(p_pc) else np.nan
            r_i = float(r_pc[i]) if i < len(r_pc) else np.nan
            f1_i = (2 * p_i * r_i / (p_i + r_i)) if (p_i + r_i) > 0 else np.nan
            per_class_rows.append({
                "class":    cls_name,
                "class_id": int(cls_idx),
                "AP50":     round(float(ap50_pc[i]),   4),
                "AP50-95":  round(float(ap5095_pc[i]), 4),
                "precision":round(p_i,  4),
                "recall":   round(r_i,  4),
                "F1":       round(f1_i, 4) if not np.isnan(f1_i) else np.nan,
            })

    per_class_df = pd.DataFrame(per_class_rows).sort_values("class_id").reset_index(drop=True)
    print("\nPer-class metrics (mask):")
    display(per_class_df)

    # Bar chart of AP50, AP50-95 and F1; colours come from CLASS_COLORS_RGB
    # (defined in the Evaluation Widget cell).
    bar_metrics = ["AP50", "AP50-95", "F1"]
    bar_colors  = [
        tuple(v/255 for v in CLASS_COLORS_RGB.get(int(r["class_id"]), (150,150,150)))
        for _, r in per_class_df.iterrows()
    ]
    fig_pc, axes_pc = plt.subplots(1, 3, figsize=(13, 4))
    fig_pc.suptitle("Per-class test metrics (mask)", fontsize=11)
    for ax, bm in zip(axes_pc, bar_metrics):
        vals = per_class_df[bm].values.astype(float)
        ax.bar(per_class_df["class"], vals, color=bar_colors)
        for j, v in enumerate(vals):
            if not np.isnan(v):
                ax.text(j, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
        ax.set_title(bm); ax.set_ylim(0, 1.1)
        ax.set_ylabel("Score"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    perclass_png = RESULTS_DIR / f"{PROJECT_NAME}_per_class_metrics.png"
    fig_pc.savefig(perclass_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", perclass_png)

except Exception as e:
    print(f"Per-class metrics not available: {e}")
    per_class_df = pd.DataFrame()

# --- Inference speed metrics ---
speed_raw  = getattr(eval_results, "speed", {}) or {}

# Speed values are ms per image; key names differ between Ultralytics versions.
def _spd(key_patterns):
    for k in speed_raw:
        for p in key_patterns:
            if p.lower() in k.lower():
                return float(speed_raw[k])
    return np.nan

speed_metrics = {
    "preprocess_ms":  _spd(["preprocess"]),
    "inference_ms":   _spd(["inference"]),
    "postprocess_ms": _spd(["postprocess"]),
}
speed_metrics["total_ms"] = sum(
    v for v in speed_metrics.values() if not np.isnan(v)
)
speed_metrics["fps"] = round(1000 / speed_metrics["total_ms"], 1) \
    if speed_metrics["total_ms"] > 0 else np.nan

print("\nInference speed (ms / image):")
display(pd.DataFrame([speed_metrics]).round(3))

# Speed bar chart
fig_spd, axes_spd = plt.subplots(1, 2, figsize=(10, 4))
fig_spd.suptitle(f"Inference speed on test set — {SELECTED_MODEL}", fontsize=11)

stage_labels = ["preprocess_ms", "inference_ms", "postprocess_ms"]
stage_vals   = [speed_metrics[k] for k in stage_labels]
stage_colors = ["tab:blue", "tab:orange", "tab:green"]

ax_bar = axes_spd[0]
bars   = ax_bar.bar(
    [l.replace("_ms", "") for l in stage_labels],
    stage_vals, color=stage_colors,
)
for bar, v in zip(bars, stage_vals):
    if not np.isnan(v):
        ax_bar.text(bar.get_x() + bar.get_width()/2, v + 0.02,
                    f"{v:.2f} ms", ha="center", va="bottom", fontsize=9)
ax_bar.set_ylabel("ms / image"); ax_bar.set_title("Latency breakdown")
ax_bar.grid(axis="y", alpha=0.3)

ax_fps = axes_spd[1]
ax_fps.bar(["FPS"], [speed_metrics["fps"]], color="tab:purple")
ax_fps.text(0, speed_metrics["fps"] + 0.5,
            f"{speed_metrics['fps']:.1f}", ha="center", va="bottom", fontsize=11)
ax_fps.set_ylabel("Frames / second"); ax_fps.set_title("Throughput")
ax_fps.grid(axis="y", alpha=0.3)

plt.tight_layout()
speed_png = RESULTS_DIR / f"{PROJECT_NAME}_inference_speed.png"
fig_spd.savefig(speed_png, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", speed_png)

# --- Summary CSV ---
summary = {
    "model":    SELECTED_MODEL,
    "run_name": RUN_NAME,
    "weights":  str(FINAL_WEIGHTS_BEST),
}
summary.update({f"test_{k}": v for k, v in agg_metrics.items()})
summary.update({f"speed_{k}": v for k, v in speed_metrics.items()})

# Best validation value and epoch per metric from results.csv
if FINAL_RESULTS_CSV.exists():
    for lbl, pats in metric_specs:
        col = _find_col(tdf, pats)
        if col:
            vals = pd.to_numeric(tdf[col], errors="coerce")
            summary[f"best_val_{lbl.lower().replace(' ','_').replace('@','')}"] = round(float(vals.max()), 4)
            summary[f"best_val_{lbl.lower().replace(' ','_').replace('@','')}_epoch"] = int(ep[vals.idxmax()])

pd.DataFrame([summary]).to_csv(FINAL_EVAL_CSV, index=False)
print("\nSaved full summary CSV to:", FINAL_EVAL_CSV)

# Per-class CSV
if not per_class_df.empty:
    pc_csv = RESULTS_DIR / f"{PROJECT_NAME}_per_class_metrics.csv"
    per_class_df.to_csv(pc_csv, index=False)
    print("Saved per-class CSV to:    ", pc_csv)

In [ ]:
# === Cell 10: Evaluation Widget ===

# Interactive viewer for val/test images: predictions (solid) and ground truth
# (dashed) with adjustable confidence and IoU thresholds.

import math, random
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Color palettes ---
CLASS_COLORS_RGB = {          # prediction colors  (solid, vivid)
    0: (255,   0,   0),       # body  — red
    1: (  0, 200,   0),       # cap   — green
    2: (  0,  80, 255),       # label — blue
}
GT_COLORS_RGB = {             # ground-truth colors (lighter pastels)
    0: (255, 160, 160),       # body  — light pink/red
    1: (160, 255, 160),       # cap   — light green
    2: (160, 200, 255),       # label — light blue
}
DEFAULT_COLOR_RGB  = (255, 255,   0)
DEFAULT_GT_RGB     = (255, 255, 160)
MASK_ALPHA         = 0.35
BOX_THICKNESS      = 2
TEXT_SCALE         = 0.55
TEXT_THICKNESS     = 2

CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

def _rgb2bgr(c):  return (c[2], c[1], c[0])
def _cls_bgr(cid):   return _rgb2bgr(CLASS_COLORS_RGB.get(int(cid), DEFAULT_COLOR_RGB))
def _gt_bgr(cid):    return _rgb2bgr(GT_COLORS_RGB.get(int(cid), DEFAULT_GT_RGB))

# --- Dashed polygon helper (OpenCV has no dashed line style) ---
def draw_dashed_polygon(img, points, color, thickness=2, dash_px=12, gap_px=7):
    """Draw a closed dashed polygon on img in-place."""
    pts = list(points)
    pts.append(pts[0])                          # close the loop
    for i in range(len(pts) - 1):
        x0, y0 = int(pts[i][0]),   int(pts[i][1])
        x1, y1 = int(pts[i+1][0]), int(pts[i+1][1])
        dx, dy  = x1 - x0, y1 - y0
        seg_len = max(1, int(np.hypot(dx, dy)))
        step    = dash_px + gap_px
        t       = 0
        while t < seg_len:
            t_end   = min(t + dash_px, seg_len)
            sx = int(round(x0 + dx * t     / seg_len))
            sy = int(round(y0 + dy * t     / seg_len))
            ex = int(round(x0 + dx * t_end / seg_len))
            ey = int(round(y0 + dy * t_end / seg_len))
            cv2.line(img, (sx, sy), (ex, ey), color, thickness, cv2.LINE_AA)
            t += step

# --- Renderer ---
def draw_overlay(image_path, result, show_masks=True, show_contours=True,
                 show_boxes=False, show_labels=True, show_gt=True):
    """
    Render predictions (solid contours) and GT (dashed, lighter lines) on one image.
    GT polygons are read from the YOLO label files in LABELS_DIR.
    """
    img_bgr = result.orig_img.copy()
    overlay  = img_bgr.copy()
    boxes    = result.boxes
    masks    = result.masks
    h, w     = img_bgr.shape[:2]

    # --- Filled prediction masks ---
    if show_masks and masks is not None and masks.xy is not None:
        for i, poly in enumerate(masks.xy):
            if poly is None or len(poly) < 3: continue
            cls_id = int(boxes.cls[i].item()) if boxes is not None and i < len(boxes) else -1
            pts    = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], _cls_bgr(cls_id))
    blended = cv2.addWeighted(overlay, MASK_ALPHA, img_bgr, 1 - MASK_ALPHA, 0)

    # --- Solid prediction contours, boxes and labels ---
    if boxes is not None and len(boxes):
        xyxy    = boxes.xyxy.cpu().numpy()
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs   = boxes.conf.cpu().numpy()
        polys   = masks.xy if masks is not None and masks.xy is not None else []

        for i in range(len(xyxy)):
            cls_id     = int(cls_ids[i])
            conf       = float(confs[i])
            color      = _cls_bgr(cls_id)
            x1,y1,x2,y2 = xyxy[i].astype(int)

            if show_contours and i < len(polys):
                poly = polys[i]
                if poly is not None and len(poly) >= 3:
                    pts = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
                    cv2.polylines(blended, [pts], True, color, 2, cv2.LINE_AA)

            if show_boxes:
                cv2.rectangle(blended, (x1,y1), (x2,y2), color, BOX_THICKNESS)

            if show_labels:
                name  = CLASS_ID_TO_NAME.get(cls_id, str(cls_id))
                label = f"{name} {conf:.2f}"
                (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX,
                                               TEXT_SCALE, TEXT_THICKNESS)
                ly1 = max(0, y1 - th - bl - 4)
                cv2.rectangle(blended, (x1, ly1), (x1 + tw + 6, y1), color, -1)
                cv2.putText(blended, label, (x1 + 3, y1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE,
                            (255,255,255), TEXT_THICKNESS, cv2.LINE_AA)

    # --- Dashed GT polygon outlines ---
    if show_gt:
        label_path = LABELS_DIR / (Path(image_path).stem + ".txt")
        if label_path.exists():
            with label_path.open("r", encoding="utf-8") as f:
                for line in f:
                    parsed = parse_yolo_label_line(line)
                    if parsed is None: continue
                    pts_px = [
                        (int(round(x * w)), int(round(y * h)))
                        for x, y in parsed["polygon_norm"]
                    ]
                    if len(pts_px) < 3: continue
                    gt_color = _gt_bgr(parsed["class_id"])
                    draw_dashed_polygon(blended, pts_px, gt_color,
                                        thickness=2, dash_px=12, gap_px=7)

    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)

# --- Val and test image pools ---
val_pool  = [Path(p.strip()) for p in FINAL_VAL_TXT.read_text(encoding="utf-8").splitlines()  if p.strip()]
test_pool = [Path(p.strip()) for p in FINAL_TEST_TXT.read_text(encoding="utf-8").splitlines() if p.strip()]

if not val_pool and not test_pool:
    print("No images found in val or test split.")
else:
    if "FINAL_WEIGHTS_BEST" not in globals() or not FINAL_WEIGHTS_BEST.exists():
        raise FileNotFoundError("Best weights not found. Run Cell 7 first.")
    viz_model = YOLO(str(FINAL_WEIGHTS_BEST))

    mode_toggle = widgets.ToggleButtons(
        options=[("Val images", "val"), ("Test images", "test")],
        value="val" if val_pool else "test",
        description="Mode:", button_style="",
    )

    def _make_opts(pool):
        return [(f"{i}: {p.name}", i) for i, p in enumerate(pool)]

    _init_pool     = val_pool if mode_toggle.value == "val" else test_pool
    image_dropdown = widgets.Dropdown(
        options=_make_opts(_init_pool), description="Image:",
        layout=widgets.Layout(width="700px"),
    )

    show_button   = widgets.Button(description="Show",    button_style="primary", icon="eye")
    random_button = widgets.Button(description="Random",  button_style="info",    icon="random")
    save_button   = widgets.Button(description="Save PNG",                         icon="save")

    conf_slider = widgets.FloatSlider(
        value=CONF_THRESHOLD, min=0.0, max=1.0, step=0.01,
        description="Conf:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )
    iou_slider = widgets.FloatSlider(
        value=IOU_THRESHOLD, min=0.10, max=0.90, step=0.05,
        description="IoU:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )

    show_masks_cb    = widgets.Checkbox(value=True,  description="Masks")
    show_contours_cb = widgets.Checkbox(value=True,  description="Contours")
    show_boxes_cb    = widgets.Checkbox(value=False, description="Boxes")
    show_labels_cb   = widgets.Checkbox(value=True,  description="Labels")
    show_gt_cb       = widgets.Checkbox(value=True,  description="Ground truth (dashed)")

    viewer_output = widgets.Output()

    def _current_pool():   return val_pool  if mode_toggle.value == "val" else test_pool
    def _current_prefix(): return "Val"     if mode_toggle.value == "val" else "Test"

    def _render(save_path=None):
        pool = _current_pool()
        idx  = image_dropdown.value
        if idx is None or idx < 0 or idx >= len(pool): return
        image_path = pool[idx]
        results = viz_model.predict(
            source=str(image_path), conf=conf_slider.value,
            iou=iou_slider.value, imgsz=IMAGE_SIZE,
            retina_masks=True, verbose=False, save=False,
        )
        r      = results[0]
        n_pred = 0 if r.boxes is None else len(r.boxes)
        rgb    = draw_overlay(
            image_path, r,
            show_masks=show_masks_cb.value, show_contours=show_contours_cb.value,
            show_boxes=show_boxes_cb.value, show_labels=show_labels_cb.value,
            show_gt=show_gt_cb.value,
        )
        with viewer_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.imshow(rgb)
            ax.set_title(
                f"{_current_prefix()}: {image_path.name}  |  pred={n_pred}  |  "
                f"conf≥{conf_slider.value:.2f}\n"
                f"solid=prediction  dashed=GT  |  "
                f"body=red  cap=green  label=blue  (GT = lighter shade)",
                fontsize=9,
            )
            ax.axis("off")
            plt.tight_layout()
            if save_path:
                fig.savefig(save_path, bbox_inches="tight", dpi=150)
                print(f"Saved: {save_path}")
            plt.show()

    def on_mode_changed(change):
        if change["name"] != "value": return
        pool = _current_pool()
        image_dropdown.options = _make_opts(pool)
        if pool: image_dropdown.value = 0

    mode_toggle.observe(on_mode_changed)
    show_button.on_click(lambda _: _render())
    random_button.on_click(lambda _: (
        setattr(image_dropdown, "value", random.randrange(len(_current_pool()))),
        _render(),
    ))
    save_button.on_click(lambda _: _render(
        save_path=RESULTS_DIR / f"viz_{'val' if mode_toggle.value == 'val' else 'test'}"
                                f"_{Path(_current_pool()[image_dropdown.value]).stem}.png"
    ))

    display(
        widgets.VBox([
            widgets.HBox([mode_toggle]),
            widgets.HBox([image_dropdown]),
            widgets.HBox([show_button, random_button, save_button]),
            widgets.HBox([conf_slider, iou_slider]),
            widgets.HBox([show_masks_cb, show_contours_cb, show_boxes_cb,
                          show_labels_cb, show_gt_cb]),
        ]),
        viewer_output,
    )
    on_mode_changed({"name": "value", "new": mode_toggle.value})

In [ ]:
# === Cell 11: Per-Class AP50-95 Comparison ===
# Grouped bar chart of per-class mask AP50-95 for YOLOv8n and YOLO26n, read from the
# per-class CSVs of both final-training notebooks (thesis figure fig:res-seg-perclass).
# Requires only KI_SORT_ROOT from the Data Root cell.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# --- Paths ---
CSV_V8N = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation"
               "/20260529_Yolo_Final_Training_yolov8n/results"
               "/20260529_Yolo_Final_Training_yolov8n_per_class_metrics.csv")
CSV_26N = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation"
               "/20260529_Yolo_Final_Training_yolo26n/results"
               "/20260529_Yolo_Final_Training_yolo26n_per_class_metrics.csv")

SAVE_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation"
                "/20260529_Yolo_Final_Training_yolov8n/results")

# --- Plot style ---
PREFERRED_SANS_FONTS = ["DejaVu Sans",]
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_selected_font = next((n for n in PREFERRED_SANS_FONTS if n in _available_fonts), "DejaVu Sans")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": PREFERRED_SANS_FONTS,
    "font.size": 10,
    "mathtext.fontset": "dejavusans",
})

_FW           = 16 / 2.54          # LaTeX \textwidth (16 cm) in inches
_FONT_SIZE    = 10
_BAND_GREY    = "#f0f0f0"
_YLIM_HEADROOM = 1.18
_DPI          = 150

# Model colours (tab10), as in the model screening notebook
MODEL_COLORS = {
    "YOLOv8n": "#1f77b4",
    "YOLO26n": "#e377c2",
}

# Class display order
CLASS_ORDER = ["body", "cap", "label"]

# --- Metric column ---
METRIC_COL = "AP50-95"             # column name in the per-class CSVs


def _add_bg_bands(ax, n):
    """Alternating light-grey / white vertical bands behind categorical bars."""
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=_BAND_GREY, zorder=0)


# --- Load and merge ---
for p in (CSV_V8N, CSV_26N):
    assert p.exists(), f"CSV not found: {p}"

df_v8n = pd.read_csv(CSV_V8N)
df_26n = pd.read_csv(CSV_26N)

print("YOLOv8n CSV columns:", list(df_v8n.columns))
print("YOLO26n CSV columns:", list(df_26n.columns))

# Normalise class names to lowercase
df_v8n["class"] = df_v8n["class"].astype(str).str.strip().str.lower()
df_26n["class"] = df_26n["class"].astype(str).str.strip().str.lower()

# Average multiple rows per class (e.g. per fold), if present
df_v8n = df_v8n.groupby("class")[METRIC_COL].mean().reset_index()
df_26n = df_26n.groupby("class")[METRIC_COL].mean().reset_index()

# Filter to the three target classes in display order
df_v8n = df_v8n[df_v8n["class"].isin(CLASS_ORDER)].set_index("class").loc[CLASS_ORDER].reset_index()
df_26n = df_26n[df_26n["class"].isin(CLASS_ORDER)].set_index("class").loc[CLASS_ORDER].reset_index()

print(f"\nYOLOv8n per-class {METRIC_COL}:")
print(df_v8n.to_string(index=False))
print(f"\nYOLO26n per-class {METRIC_COL}:")
print(df_26n.to_string(index=False))

# --- Plot ---
n_classes = len(CLASS_ORDER)
x = np.arange(n_classes)
bar_width = 0.32

fig, ax = plt.subplots(figsize=(_FW, _FW * 0.55))

# Background bands
_add_bg_bands(ax, n_classes)

# Bars
bars_v8n = ax.bar(
    x - bar_width / 2,
    df_v8n[METRIC_COL].values,
    width=bar_width,
    color=MODEL_COLORS["YOLOv8n"],
    edgecolor="black",
    linewidth=0.6,
    label="YOLOv8n",
    zorder=3,
)
bars_26n = ax.bar(
    x + bar_width / 2,
    df_26n[METRIC_COL].values,
    width=bar_width,
    color=MODEL_COLORS["YOLO26n"],
    edgecolor="black",
    linewidth=0.6,
    label="YOLO26n",
    zorder=3,
)

# Bold value labels above each bar
for bars in (bars_v8n, bars_26n):
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.008,
            f"{h:.3f}",
            ha="center", va="bottom",
            fontsize=_FONT_SIZE - 1,
            fontweight="bold",
            zorder=4,
        )

# Axes
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in CLASS_ORDER], fontsize=_FONT_SIZE)
ax.set_ylabel(r"$\mathrm{AP}_{50\text{-}95}^{(M)}$", fontsize=_FONT_SIZE)
ax.set_title(r"Per-Class $\mathrm{AP}_{50\text{-}95}^{(M)}$ on Test Set", fontsize=_FONT_SIZE + 1)

y_max = max(df_v8n[METRIC_COL].max(), df_26n[METRIC_COL].max())
ax.set_ylim(0, y_max * _YLIM_HEADROOM)
ax.set_xlim(-0.5, n_classes - 0.5)

ax.legend(loc="best", fontsize=_FONT_SIZE, framealpha=0.9)
ax.grid(axis="y", linewidth=0.4, alpha=0.6)
ax.set_axisbelow(True)

plt.tight_layout()

# --- Save ---
save_path = SAVE_DIR / "20260529_Yolo_Final_Training_per_class_comparison.png"
fig.savefig(save_path, dpi=_DPI, bbox_inches="tight")
print(f"\nSaved → {save_path}")
plt.show()
